In [1]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep

pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

import warnings
warnings.filterwarnings('ignore')

In [ ]:
precos_teto_suno_dividendos = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
}

precos_teto_suno_valor = {"VAMO3": 10.90, "B3SA3": 17.00, "KLBN11": 25.00, "TTEN3": 14.70, "PRIO3": 62.75, "BRBI11": 18.00,
    "PNVL3": 12.00, "SIMH3": 10.00, "GMAT3": 7.12, "TIMS3": 18.60, "VIVA3": 25.00, "EZTC3": 13.29,
    "BRKM5": 999,
}



carteira_PM = {
    "ABEV3": 10.00, "B3SA3": 11.08, "BBAS3": 21.96, "BBSE3": 32.00,
    "EGIE3": 28, "FLRY3": 15.60, "HYPE3": 29.31, "ITSA4": 9.41,
    "KLBN11": 18.58, "LEVE3": 33.92, "PETR4": 30.06, "TAEE11": 38.28,
    "UNIP6": 50.99, "VALE3": 58.13, "RADL3":25
}

# Valuation Gemini - Fevereiro/Março 2026
# Metodologia: Crescimento de Lucro (CAGR) + Múltiplos Setoriais
# Premissas: Juros 9% a.a. | Margem de Segurança: 30%

valuation_gemini_9pct = {
    "VALE3": 75.60, "PETR4": 35.00, "BBAS3": 40.95, "BBSE3": 32.20,
    "PRIO3": 50.40, "ITSA4": 12.04, "EGIE3": 37.80, "TAEE11": 30.45,
    "ABEV3": 12.25, "LEVE3": 35.70, "HYPE3": 30.80, "FLRY3": 16.45,
    "KLBN11": 21.00, "UNIP6": 71.40, "B3SA3": 11.34, "TTEN3": 13.65,
    "VAMO3": 9.24, "SIMH3": 8.26, "GMAT3": 8.05, "EZTC3": 18.20,
    "TUPY3": 26.95, "AGRO3": 23.80, "TIMS3": 15.96, "VIVA3": 23.10,
    "WIZC3": 8.54, "BRKM5": 25.20, "PNVL3": 10.85, "SEER3": 8.96,
    "BRBI11": 18.90, "AXIA6": 33.60, "DEXP4":10.15
}

valuation_perene_extra = {
    # Saneamento
    "SBSP3": 80.50, "SAPR11": 26.60, "CSMG3": 19.60,
    # Transmissão e Agro
    "TRPL4": 25.20, "SLCE3": 18.20, 
    # Logística e Adições de Valor
    "STBP3": 12.95, "SUZB3": 54.60, "MDIA3": 31.50
}

precos_teto = {}
for k, v in carteira_PM.items():
    precos_teto[k] = precos_teto_suno_dividendos.get(k, v)

# precos_teto = precos_teto_suno

# precos_teto.update(valuation_perene_extra)



In [3]:
# ---------------------------------------------------
# CRIAR PASTA
# ---------------------------------------------------

if not os.path.exists("dbJson"):
    os.makedirs("dbJson")


# ---------------------------------------------------
# BAIXAR DADOS
# ---------------------------------------------------

def atualizarDados(ativo):

    url = f"https://stock-options-manager-prod.firebaseapp.com/api/option/{ativo.upper()}"
           

    response = requests.get(url)

    if response.status_code != 200:
        print(f"{ativo}: erro HTTP {response.status_code}")
        return

    try:
        dados = response.json()
    except Exception as e:
        print(f"{ativo}: erro JSON -> {e}")
        return

    with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
        json.dump(dados, arq, indent=2)


for ativo in precos_teto.keys():

    atualizarDados(ativo)
    # sleep(3)

    print(f"{ativo}: atualizado")

ABEV3: atualizado
B3SA3: atualizado
BBAS3: atualizado
BBSE3: atualizado
EGIE3: atualizado
FLRY3: atualizado
HYPE3: atualizado
ITSA4: atualizado
KLBN11: atualizado
LEVE3: atualizado
PETR4: atualizado
TAEE11: atualizado
UNIP6: atualizado
VALE3: atualizado
RADL3: atualizado


In [4]:
# ---------------------------------------------------
# CRIAR DATAFRAME
# ---------------------------------------------------

def dataFrameUnico(ativo):

    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]

    linhas = []

    for serie in dados["series"]:

        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:

            strike_price = strike["strike"]

            # CALL
            if strike["call"]:

                c = strike["call"]

                linhas.append({
                    "ativo": ativo,
                    "tipo": "CALL",
                    "vencimento": vencimento,
                    "dias": dias,
                    "strike": strike_price,
                    "symbol": c["symbol"],
                    "bid": c["bid"],
                    "ask": c["ask"],
                    "volume": c["volume"],
                    "delta": c["bs"]["delta"],
                    "theta": c["bs"]["theta"],
                    "vol": c["bs"]["volatility"],
                    "poe": c["bs"]["poe"],
                    "preco_atual": preco_atual
                })

            # PUT
            if strike["put"]:

                p = strike["put"]

                linhas.append({
                    "ativo": ativo,
                    "tipo": "PUT",
                    "vencimento": vencimento,
                    "dias": dias,
                    "strike": strike_price,
                    "symbol": p["symbol"],
                    "bid": p["bid"],
                    "ask": p["ask"],
                    "volume": p["volume"],
                    "delta": p["bs"]["delta"],
                    "theta": p["bs"]["theta"],
                    "vol": p["bs"]["volatility"],
                    "poe": p["bs"]["poe"],
                    "preco_atual": preco_atual
                })

    df = pd.DataFrame(linhas)

    return df





# ---------------------------------------------------
# CRIAR DATAFRAME GERAL
# ---------------------------------------------------

todos = []

for ativo, preco_teto in precos_teto.items():

    try:

        df = dataFrameUnico(ativo)

        df["preco_teto"] = preco_teto

        todos.append(df)

    except Exception as e:

        print(f"Erro em {ativo}: {e}")
        pass


df_final = pd.concat(todos, ignore_index=True)

In [5]:
# ------------------------------------------
# FILTRO PARA VENDA DE PUT
# ------------------------------------------
puts = df_final[df_final["tipo"] == "PUT"]

puts["retorno"] = (puts["bid"]/ (puts["strike"] - puts['bid']))  # Retorno ROR
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"])

puts['retorno'] = round(puts['retorno'] * 100,2)
puts['retorno_mes'] = round(puts['retorno_mes'] * 100,2)


puts["dist_strike"] = round((puts["strike"] / puts["preco_atual"] - 1) * 100,2)

# ------------------------------------------
# APORTE
# ------------------------------------------

aporte = 5550

puts['cotas'] = np.floor((aporte / puts['strike'])/100) * 100
puts['premio X cotas'] = puts['cotas'] * puts['bid']

# ------------------------------------------
# BLACK scholes
# ------------------------------------------

import numpy as np
from scipy.stats import norm

r = 0.1475  # taxa livre de risco

T = puts['dias'] / 252

# volatilidade em decimal
sigma = puts['vol'] / 100

d1 = (
    np.log(puts['preco_atual'] / puts['strike']) +
    (r + sigma**2 / 2) * T
) / (sigma * np.sqrt(T))

d2 = d1 - sigma * np.sqrt(T)

puts['black_scholes'] = round((
    puts['strike'] * np.exp(-r * T) * norm.cdf(-d2)
    - puts['preco_atual'] * norm.cdf(-d1)
),2)

puts['desvio_bs'] = round( puts['bid'] - puts['black_scholes'],2)




In [6]:
puts

,ativo,tipo,vencimento,dias,strike,symbol,bid,ask,volume,delta,theta,vol,poe,preco_atual,preco_teto,retorno,retorno_mes,dist_strike,cotas,premio X cotas,black_scholes,desvio_bs
1,ABEV3,PUT,2026-03-27,1,10.75,ABEVO107W4,0.00,0.00,0,0.000000,0.000000,0.000,0.00,14.94,10.0,0.00,0.00,-28.05,500.0,0.0,0.00,0.00
3,ABEV3,PUT,2026-03-27,1,11.25,ABEVO112W4,0.00,0.00,0,0.000000,0.000000,0.000,0.00,14.94,10.0,0.00,0.00,-24.70,400.0,0.0,0.00,0.00
5,ABEV3,PUT,2026-03-27,1,11.75,ABEVO117W4,0.00,0.00,0,0.000000,0.000000,0.000,0.00,14.94,10.0,0.00,0.00,-21.35,400.0,0.0,0.00,0.00
7,ABEV3,PUT,2026-03-27,1,12.25,ABEVO122W4,0.00,0.01,0,0.000000,0.000000,154.816,0.00,14.94,10.0,0.00,0.00,-18.01,400.0,0.0,0.01,-0.01
9,ABEV3,PUT,2026-03-27,1,12.75,ABEVO127W4,0.00,0.01,0,0.000000,0.000000,143.963,0.00,14.94,10.0,0.00,0.00,-14.66,400.0,0.0,0.02,-0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17445,RADL3,PUT,2027-02-19,225,36.19,RADLN361,0.00,0.00,0,-0.723074,0.009052,0.000,83.16,23.80,25.0,0.00,0.00,52.06,100.0,0.0,7.92,-7.92
17447,RADL3,PUT,2027-04-16,264,20.27,RADLP209,0.01,0.00,0,-0.160400,-0.001551,37.379,27.64,23.80,25.0,0.05,0.01,-14.83,200.0,2.0,0.89,-0.88
17449,RADL3,PUT,2027-08-20,352,19.46,RADLT200,0.01,0.00,0,-0.131617,-0.000939,38.625,25.53,23.80,25.0,0.05,0.00,-18.24,200.0,2.0,0.89,-0.88
17451,RADL3,PUT,2027-12-17,433,22.00,RADLX220,0.00,0.00,2000,-0.181347,-0.000230,39.261,34.49,23.80,25.0,0.00,0.00,-7.56,200.0,0.0,1.60,-1.60


In [7]:
filtro = puts[

    (puts['dias'].between(1, 90)) &
    (puts['dist_strike'] <= -5) &
    (puts['retorno_mes'] >= 1) & # maior que o cdi liquido mensal
    (puts['retorno'] >= 1) &
    (puts['delta'] >= -0.30 )&
    (puts['poe'] <= 30) &
    (puts['strike'] <= puts['preco_teto'] * 1.05) &
    # (puts['premio X cotas'] >= 80) &
    
    (puts['volume'] > 0)

].copy()

filtro['rank_retorno_mes'] = filtro['retorno_mes'].rank(ascending=False)
filtro['rank_dias'] = filtro['dias'].rank(ascending=True)
filtro['rank_dist_strike'] = filtro['dist_strike'].rank(ascending=True)
filtro['rank_poe'] = filtro['poe'].rank(ascending=True)
filtro['rank_premio X cotas'] = filtro['premio X cotas'].rank(ascending=False)

filtro['score'] = (
    filtro['rank_dist_strike'] * 5 +
    filtro['rank_poe'] * 3 +
    filtro['rank_retorno_mes'] * 4 +
    filtro['rank_premio X cotas'] * 3.5 +
    filtro['rank_dias'] * 3
)

filtro[[
    'ativo', 'symbol', 'tipo',  'preco_atual', 'strike', 'dist_strike', 'bid','black_scholes','desvio_bs' ,'ask',
    'volume', 'delta', 'theta', 'vol', 'poe',  'preco_teto',
    'retorno', 'retorno_mes', 'vencimento',  'dias', 'cotas', 'premio X cotas','score'
]].sort_values('score', ascending=True)

,ativo,symbol,tipo,preco_atual,strike,dist_strike,bid,black_scholes,desvio_bs,ask,volume,delta,theta,vol,poe,preco_teto,retorno,retorno_mes,vencimento,dias,cotas,premio X cotas,score
1803,B3SA3,B3SAP163,PUT,17.85,16.37,-8.29,0.27,0.28,-0.01,0.88,52400,-0.172756,-0.014382,51.854,20.16,17.0,1.68,3.35,2026-04-17,15,300.0,81.0,86.75
2143,B3SA3,B3SAQ155,PUT,17.85,15.53,-13.00,0.27,0.32,-0.05,0.36,63600,-0.140647,-0.007992,50.526,17.92,17.0,1.77,1.61,2026-05-15,33,300.0,81.0,114.25
2147,B3SA3,B3SAQ163,PUT,17.85,16.03,-10.20,0.37,0.41,-0.04,0.44,8400,-0.189912,-0.009478,48.908,23.60,17.0,2.36,2.15,2026-05-15,33,300.0,111.0,126.50
17131,RADL3,RADLP225,PUT,23.80,22.15,-6.93,0.32,0.33,-0.01,0.36,232950,-0.185157,-0.017413,44.564,21.16,25.0,1.47,2.93,2026-04-17,15,200.0,64.0,130.50
2145,B3SA3,B3SAQ157,PUT,17.85,15.78,-11.60,0.24,0.36,-0.12,0.45,3100,-0.164277,-0.008758,49.564,20.67,17.0,1.54,1.40,2026-05-15,33,300.0,72.0,141.00
2379,B3SA3,B3SAR160,PUT,17.85,15.28,-14.40,0.32,0.30,0.02,0.00,300,-0.156414,-0.005998,42.116,21.18,17.0,2.14,1.13,2026-06-19,57,300.0,96.0,145.00
1805,B3SA3,B3SAP166,PUT,17.85,16.62,-6.89,0.25,0.33,-0.08,0.45,24200,-0.211273,-0.016084,50.714,24.36,17.0,1.53,3.05,2026-04-17,15,300.0,75.0,148.00
1807,B3SA3,B3SAP168,PUT,17.85,16.87,-5.49,0.31,0.40,-0.09,0.43,449000,-0.253727,-0.017578,50.426,28.93,17.0,1.87,3.74,2026-04-17,15,300.0,93.0,153.50
2151,B3SA3,B3SAQ165,PUT,17.85,16.53,-7.39,0.40,0.54,-0.14,0.80,9200,-0.246628,-0.010703,48.233,29.94,17.0,2.48,2.25,2026-05-15,33,300.0,120.0,154.00
2149,B3SA3,B3SAQ162,PUT,17.85,16.28,-8.80,0.35,0.47,-0.12,0.90,13100,-0.217419,-0.010132,48.483,26.70,17.0,2.20,2.00,2026-05-15,33,300.0,105.0,157.00


In [8]:
puts["retorno"] = (puts["bid"]/ (puts["strike"] - puts['bid']))  # Retorno ROR
puts["retorno_anual"] = puts["retorno"] * (365 / puts["dias"])

puts['retorno'] = round(puts['retorno'] * 100,2)
puts['retorno_anual'] = round(puts['retorno_anual'] * 100,2)


In [9]:
# PUT LONGA

puts[
    # (puts['dias'].between(90,720)) &
    (puts['volume'] > 0 ) & 
    # (puts['retorno_anual'] >= 14)
    (puts['black_scholes'] >= 2) &
    # (puts['bid'] == 0) &
    (puts['dist_strike'] <= -5)
][[
'ativo', 'symbol', 'tipo',  'preco_atual', 'strike', 'dist_strike', 'bid','black_scholes','desvio_bs' ,'ask',
    'volume', 'delta', 'theta', 'vol', 'poe',  'preco_teto',
    'retorno', 'retorno_anual', 'vencimento',  'dias', 'cotas', 'premio X cotas'
]].sort_values('retorno_anual', ascending= False)



,ativo,symbol,tipo,preco_atual,strike,dist_strike,bid,black_scholes,desvio_bs,ask,volume,delta,theta,vol,poe,preco_teto,retorno,retorno_anual,vencimento,dias,cotas,premio X cotas
12877,PETR4,PETRM45,PUT,48.02,45.50,-5.25,3.40,3.42,-0.02,3.61,100,-0.248260,-0.004047,41.724,36.56,34.0,8.08,14.74,2027-01-15,200,100.0,340.0
12771,PETR4,PETRX451,PUT,48.02,43.72,-8.95,2.50,2.55,-0.05,0.00,100,-0.214981,-0.004873,40.233,32.04,34.0,6.07,12.03,2026-12-18,184,100.0,250.0
15823,VALE3,VALET800,PUT,78.93,72.46,-8.20,2.09,2.10,-0.01,2.50,10600,-0.203636,-0.011234,32.140,26.48,75.0,2.97,10.63,2026-08-21,102,0.0,0.0
12917,PETR4,PETRO48,PUT,48.02,45.00,-6.29,2.50,2.97,-0.47,0.00,800,-0.226718,-0.002866,38.434,35.29,34.0,5.88,8.76,2027-03-19,245,100.0,250.0
11897,PETR4,PETRS45,PUT,48.02,44.98,-6.33,0.70,2.02,-1.32,2.25,200,-0.262449,-0.012709,40.393,33.47,34.0,1.58,7.49,2026-07-17,77,100.0,70.0
13125,PETR4,PETRN429,PUT,48.02,43.00,-10.45,2.31,3.36,-1.05,0.00,700,-0.155156,-0.000292,42.204,31.01,34.0,5.68,4.35,2028-02-18,476,100.0,231.0
16555,VALE3,VALEX800,PUT,78.93,72.46,-8.20,1.00,3.06,-2.06,4.28,100,-0.195649,-0.005242,33.613,27.81,75.0,1.40,2.78,2026-12-18,184,0.0,0.0
12963,PETR4,PETRS450,PUT,48.02,43.55,-9.31,0.01,2.51,-2.50,0.00,3000,-0.186271,-0.001628,36.870,32.22,34.0,0.02,0.03,2027-07-16,327,100.0,1.0
12767,PETR4,PETRX417,PUT,48.02,41.72,-13.12,0.00,2.16,-2.16,0.00,2843200,-0.175054,-0.004874,41.800,27.04,34.0,0.00,0.00,2026-12-18,184,100.0,0.0
12589,PETR4,PETRV512,PUT,48.02,45.42,-5.41,0.00,2.86,-2.86,5.00,1000,-0.263491,-0.006797,40.617,36.27,34.0,0.00,0.00,2026-10-16,140,100.0,0.0


# CALLS

In [10]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------------------------------------
# FILTRAR CALLS
# ------------------------------------------

calls = df_final[df_final["tipo"] == "CALL"].copy()

# remover dados inválidos
calls = calls[
    (calls["preco_atual"] > 0) &
    (calls["strike"] > 0) &
    (calls["dias"] > 0) &
    (calls["vol"] > 0)
].copy()

# ------------------------------------------
# DISTÂNCIA DO STRIKE (%)
# ------------------------------------------

calls["dist_strike"] = (
    (calls["strike"] / calls["preco_atual"] - 1) * 100
).round(2)

# ------------------------------------------
# BLACK SCHOLES
# ------------------------------------------

r = 0.10

T = calls["dias"] / 252
sigma = calls["vol"] / 100

# evitar divisão por zero
T = T.clip(lower=0.0001)
sigma = sigma.clip(lower=0.0001)

d1 = (
    np.log(calls["preco_atual"] / calls["strike"]) +
    (r + sigma**2 / 2) * T
) / (sigma * np.sqrt(T))

d2 = d1 - sigma * np.sqrt(T)

calls["black_scholes"] = (
    calls["preco_atual"] * norm.cdf(d1) -
    calls["strike"] * np.exp(-r * T) * norm.cdf(d2)
)

calls["black_scholes"] = calls["black_scholes"].round(2)

# ------------------------------------------
# DESVIO DO PREÇO TEÓRICO
# ------------------------------------------

calls["desvio_bs"] = (
    calls["black_scholes"] - calls["ask"]
).round(2)

calls["desvio_pct"] = (
    (calls["black_scholes"] - calls["ask"]) /
    calls["black_scholes"]
) * 100

calls["desvio_pct"] = calls["desvio_pct"].round(2)

# ------------------------------------------
# PROBABILIDADE ITM
# ------------------------------------------

calls["prob_itm"] = (norm.cdf(d2) * 100).round(2)

# ------------------------------------------
# RETORNO DA CALL COBERTA
# ------------------------------------------

calls["retorno"] = (
    calls["bid"] / calls["preco_atual"]
)

calls["retorno_anual"] = (
    calls["retorno"] * (365 / calls["dias"])
)

calls["retorno"] = (calls["retorno"] * 100).round(2)
calls["retorno_anual"] = (calls["retorno_anual"] * 100).round(2)


In [26]:
# ------------------------------------------
# FILTRO DO ATIVO
# ------------------------------------------

filtro_call = calls[
    # (calls['ativo'] == 'HYPE3') &
    (calls['ask'].between(0.01,0.1)) &
    # (calls['preco_teto'] <= calls['strike']) &
    (calls['dist_strike'] >= 0) &
    # (calls['dias'].between(1,90)) &
    (calls['retorno_anual'] >= 6) &
    (calls['volume'] > 0)
].copy()

# ------------------------------------------
# RANKING
# ------------------------------------------

filtro_call['rank_retorno'] = filtro_call['retorno_anual'].rank(
    ascending=False)
filtro_call['rank_dist'] = filtro_call['dist_strike'].rank(ascending=False)
filtro_call['rank_prob'] = filtro_call['prob_itm'].rank(ascending=True)
filtro_call['rank_dias'] = filtro_call['dias'].rank(ascending=True)
filtro_call['rank_vol'] = filtro_call['volume'].rank(ascending=False)

# ------------------------------------------
# SCORE FINAL
# ------------------------------------------

filtro_call['score'] = (
    filtro_call['rank_retorno'] * 3 +
    filtro_call['rank_dist'] * 2 +
    filtro_call['rank_prob'] * 3 +
    filtro_call['rank_dias'] * 1 +
    filtro_call['rank_vol'] * 1
)

# ------------------------------------------
# RESULTADO
# ------------------------------------------

resultado = filtro_call[[
    'ativo',
    'symbol',
    'tipo',
    'bid',
    'ask',
    'preco_atual',
    'strike',
    'dist_strike',
    'prob_itm',
    'black_scholes',
    'desvio_bs',
    'desvio_pct',
    'dias',
    'vencimento',
    'preco_teto',
    'retorno',
    'retorno_anual',
    'score',
]].sort_values('score', ascending=True)

resultado

,ativo,symbol,tipo,bid,ask,preco_atual,strike,dist_strike,prob_itm,black_scholes,desvio_bs,desvio_pct,dias,vencimento,preco_teto,retorno,retorno_anual,score
9216,PETR4,PETRC500W4,CALL,0.03,0.04,48.02,50.00,4.12,5.47,0.03,-0.01,-33.33,1,2026-03-27,34.0,0.06,22.80,57.0
3034,BBAS3,BBASC244W4,CALL,0.01,0.02,23.10,24.21,4.81,6.12,0.02,0.00,0.00,1,2026-03-27,25.0,0.04,15.80,69.5
13644,VALE3,VALEC820W4,CALL,0.02,0.08,78.93,82.00,3.89,4.94,0.04,-0.04,-100.00,1,2026-03-27,75.0,0.03,9.25,70.5
9218,PETR4,PETRC505W4,CALL,0.01,0.03,48.02,50.50,5.16,3.50,0.02,-0.01,-50.00,1,2026-03-27,34.0,0.02,7.60,78.5
3032,BBAS3,BBASC242W4,CALL,0.01,0.02,23.10,23.96,3.72,7.28,0.02,0.00,0.00,1,2026-03-27,25.0,0.04,15.80,83.5
5420,BBSE3,BBSEC380W4,CALL,0.01,0.06,34.64,35.42,2.25,4.90,0.01,-0.05,-500.00,1,2026-03-27,35.5,0.03,10.54,87.0
3102,BBAS3,BBASD257W1,CALL,0.02,0.06,23.10,25.51,10.43,3.44,0.02,-0.04,-200.00,5,2026-04-02,25.0,0.09,6.32,90.5
3030,BBAS3,BBASC239W4,CALL,0.03,0.06,23.10,23.71,2.64,11.62,0.03,-0.03,-100.00,1,2026-03-27,25.0,0.13,47.40,93.0
13642,VALE3,VALEC815W4,CALL,0.02,0.09,78.93,81.50,3.26,6.49,0.05,-0.04,-80.00,1,2026-03-27,75.0,0.03,9.25,93.5
3338,BBAS3,BBASD262,CALL,0.08,0.10,23.10,26.02,12.64,8.76,0.09,-0.01,-11.11,15,2026-04-17,25.0,0.35,8.43,95.0


# A grande a aposta

In [12]:
df = df_final[df_final["tipo"] == "CALL"]

In [13]:
df[
    (df['tipo'] == 'CALL') &
    (df['ask'].between(0.01, 0.2)) &
    (df['volume'] > 0) &
    # (df['delta'].between(0.15, 0.45)) &  # alguma chance real
    (df['dias'] >= 0) &  # evita vencimento imediato
    ((df['strike'] / df['preco_atual']) <= 1.05)  # até 15% fora do dinheiro
][[
    'ativo','symbol','vencimento','dias','preco_atual',
    'strike','ask','bid','volume','delta','theta','poe'
]]

,ativo,symbol,vencimento,dias,preco_atual,strike,ask,bid,volume,delta,theta,poe
26,ABEV3,ABEVC150W4,2026-03-27,1,14.94,15.00,0.16,0.02,25100,0.428837,-0.056925,42.17
390,ABEV3,ABEVD156,2026-04-17,15,14.94,15.67,0.19,0.10,47100,0.302302,-0.014759,27.83
3030,BBAS3,BBASC239W4,2026-03-27,1,23.10,23.71,0.06,0.03,566500,0.118845,-0.050670,11.47
3032,BBAS3,BBASC242W4,2026-03-27,1,23.10,23.96,0.02,0.01,474600,0.047323,-0.025014,4.52
3034,BBAS3,BBASC244W4,2026-03-27,1,23.10,24.21,0.02,0.01,216200,0.015507,-0.009833,1.47
5420,BBSE3,BBSEC380W4,2026-03-27,1,34.64,35.42,0.06,0.01,22000,0.017490,-0.008030,1.71
5466,BBSE3,BBSED359W1,2026-04-02,5,34.64,35.93,0.09,0.00,1500,0.072964,-0.012490,6.98
7868,ITSA4,ITSAC135W4,2026-03-27,1,13.47,13.38,0.15,0.03,35400,0.646039,-0.055163,63.86
7870,ITSA4,ITSAC140W4,2026-03-27,1,13.47,13.88,0.05,0.00,37100,0.072259,-0.019110,6.95
7914,ITSA4,ITSAD141W1,2026-04-02,5,13.47,14.07,0.07,0.03,29400,0.187891,-0.017701,17.61
